# TP 1 - Architecte HDFS : Partie A (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syhard87/tp_hadloop/blob/claude/tp-hadoop-avances-partie-a-mwgfq7/notebooks/TP1_PartieA_Colab.ipynb)

Source : `sources/TP_Hadoop_avances_avec_TP_ecosysteme.pdf` (pages 5 à 9).

**Ce qui est démontré** : installation de Java 17 + SSH + Hadoop 3.5.0, NameNode et DataNode sur la même machine,
fichier de 300 Mo découpé en 3 blocs (128 + 128 + 44 Mo), réplication demandée de 3 mais une seule copie physique,
observation de la sous-réplication avec `fsck`.

**Mode d'emploi**
1. Exécuter les cellules **dans l'ordre**, de la cellule 0 à la cellule 10.
2. Ne **pas** utiliser « Réinitialiser l'environnement d'exécution » pendant le TP.
3. Si la session se déconnecte, reprendre depuis la cellule 0.

> ⚠️ **Corrections par rapport au PDF** (testées) : Colab tourne en `root`. Avec les cellules du PDF telles quelles,
> `start-dfs.sh` échoue avec `ERROR: ... there is no HDFS_NAMENODE_USER defined`. De plus, le `sed` sur `hadoop-env.sh`
> ne modifie rien, car la ligne `# export JAVA_HOME=` est commentée. La cellule 3 ajoute donc ces variables dans `hadoop-env.sh`.

## Espace partagé (Google Drive), optionnel mais recommandé

Les preuves (sorties des commandes) sont écrites dans `/content/preuves_tp`. Cette cellule monte votre Google Drive
et crée le dossier **`MyDrive/TP_Hadoop_partage`**. Partagez ce dossier avec votre binôme ou l'enseignant
(clic droit → *Partager* dans Drive). La cellule 11 y copie les preuves à la fin du TP.

In [ ]:
# Monte Google Drive (une fenêtre d'autorisation s'ouvre)
from google.colab import drive
drive.mount('/content/drive')

# Dossier partagé où seront copiées les preuves du TP
SHARED_DIR = "/content/drive/MyDrive/TP_Hadoop_partage"
import os
os.makedirs(SHARED_DIR, exist_ok=True)
print("Espace partagé :", SHARED_DIR)

## Cellule 0 - Initialiser les variables Colab

In [ ]:
# Importe le module qui permet de gérer les variables d'environnement
import os

# Version de Hadoop utilisée dans ce TP
HADOOP_VERSION = "3.5.0"

# Construit le chemin d'installation : /content/hadoop-3.5.0
HADOOP_HOME = f"/content/hadoop-{HADOOP_VERSION}"

# Indique le dossier dans lequel Java 17 est installé
JAVA_HOME = "/usr/lib/jvm/java-17-openjdk-amd64"

# Rend le chemin de Hadoop accessible aux programmes lancés dans Colab
os.environ["HADOOP_HOME"] = HADOOP_HOME

# Indique à Hadoop quelle installation Java utiliser
os.environ["JAVA_HOME"] = JAVA_HOME

# Ajoute les commandes Hadoop et les scripts d'administration au PATH
# L'ancien PATH est conservé pour garder les autres commandes du système
if f"{HADOOP_HOME}/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]

# Dossier local des preuves (checklist, point 8)
os.makedirs("/content/preuves_tp", exist_ok=True)

# Affiche les deux chemins afin de vérifier la configuration
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print("JAVA_HOME   =", os.environ["JAVA_HOME"])

## Cellule 1 - Installer les composants
Java exécute Hadoop ; OpenSSH permet aux scripts Hadoop de lancer les démons ; rsync est utilisé par certains scripts
d'administration ; wget télécharge l'archive Hadoop. **Attendu** : une version OpenJDK 17.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless openssh-server rsync wget > /dev/null
!/usr/lib/jvm/java-17-openjdk-amd64/bin/java -version

## Cellule 2 - Télécharger Hadoop 3.5.0
**Attendu** : `Hadoop 3.5.0` et les informations de compilation. Le téléchargement est ignoré s'il a déjà été fait.

In [ ]:
![ -d /content/hadoop-3.5.0 ] || (wget -q https://downloads.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz -O /content/hadoop-3.5.0.tar.gz && tar -xzf /content/hadoop-3.5.0.tar.gz -C /content)
!hadoop version

## Cellule 3 - Configurer le mode pseudo-distribué
Réplication par défaut volontairement fixée à 1 (un seul DataNode réel), bloc de 128 Mo.
**Correction** : `JAVA_HOME` et `HDFS_*_USER=root` sont ajoutés à `hadoop-env.sh`.

In [ ]:
%%bash
export HADOOP_HOME=/content/hadoop-3.5.0
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

# Correction : la ligne JAVA_HOME est commentée par défaut, le sed du PDF ne la modifie pas.
# On ajoute les variables (une seule fois) ; Colab s'exécute en root, Hadoop 3 exige HDFS_*_USER.
grep -q "TP_COLAB_FIX" $HADOOP_HOME/etc/hadoop/hadoop-env.sh || cat >> $HADOOP_HOME/etc/hadoop/hadoop-env.sh <<'EOF'
# TP_COLAB_FIX
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export HDFS_NAMENODE_USER=root
export HDFS_DATANODE_USER=root
export HDFS_SECONDARYNAMENODE_USER=root
export YARN_RESOURCEMANAGER_USER=root
export YARN_NODEMANAGER_USER=root
EOF

cat > $HADOOP_HOME/etc/hadoop/core-site.xml <<'EOF'
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>
EOF

cat > $HADOOP_HOME/etc/hadoop/hdfs-site.xml <<'EOF'
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.blocksize</name><value>134217728</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file:///content/hdfs/name</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file:///content/hdfs/data</value></property>
</configuration>
EOF

ls -l $HADOOP_HOME/etc/hadoop/core-site.xml $HADOOP_HOME/etc/hadoop/hdfs-site.xml

## Cellule 4 - Préparer SSH et démarrer HDFS
**Attendu** : `jps` affiche NameNode, DataNode, SecondaryNameNode et Jps.
Le formatage ne doit être exécuté qu'à l'initialisation de cette machine Colab temporaire.

In [ ]:
%%bash
export HADOOP_HOME=/content/hadoop-3.5.0
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH=$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH
service ssh start
mkdir -p ~/.ssh
[ -f ~/.ssh/id_rsa ] || ssh-keygen -t rsa -P '' -f ~/.ssh/id_rsa >/dev/null 2>&1
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
chmod 600 ~/.ssh/authorized_keys
ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null

# Si la cellule est rejouée, on arrête d'abord les démons avant de reformater
stop-dfs.sh >/dev/null 2>&1 || true
rm -rf /content/hdfs
hdfs namenode -format -force -nonInteractive > /content/preuves_tp/format_namenode.log 2>&1
start-dfs.sh
sleep 10
jps | tee /content/preuves_tp/jps.txt

## Cellule 5 - Vérifier le cluster Colab
**Attendu** : `Live datanodes (1)`, taille de bloc `134217728` et réplication par défaut `1`.

In [ ]:
!hdfs dfsadmin -report 2>/dev/null | tee /content/preuves_tp/dfsadmin_report.txt
!hdfs getconf -confKey dfs.blocksize
!hdfs getconf -confKey dfs.replication

## Cellule 5 bis - Diagnostic rapide en cas de problème

In [ ]:
%%bash
export HADOOP_HOME=/content/hadoop-3.5.0
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH=$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH

echo "=== Versions ==="
$JAVA_HOME/bin/java -version 2>&1 | head -n 1
hadoop version | head -n 1

echo "=== Processus Java ==="
jps

echo "=== Safe mode ==="
hdfs dfsadmin -safemode get

echo "=== DataNodes ==="
hdfs dfsadmin -report 2>/dev/null | grep -E "Live datanodes|Dead datanodes"

echo "=== Configuration ==="
echo -n "Taille de bloc : "
hdfs getconf -confKey dfs.blocksize
echo -n "Réplication par défaut : "
hdfs getconf -confKey dfs.replication

echo "=== Test écriture / lecture (checklist point 6) ==="
hdfs dfs -mkdir -p /check_colab
echo "HDFS_OK" | hdfs dfs -put -f - /check_colab/test.txt
hdfs dfs -cat /check_colab/test.txt
hdfs dfs -rm -r -f /check_colab

## Cellule 6 - Créer le fichier pratique de 300 Mo
**Attendu** : 314 572 800 octets. 300 ÷ 128 = 2,34375 → 3 blocs, dont le dernier mesure 44 Mo.

In [ ]:
!mkdir -p /content/tp1
!dd if=/dev/zero of=/content/tp1/logs_300Mo.bin bs=1M count=300 status=progress
!ls -l /content/tp1/logs_300Mo.bin

## Cellule 7 - Charger le fichier avec une réplication demandée de 3
**Attendu** : le fichier apparaît avec 300 Mo et une réplication demandée de 3, mais une seule copie physique existe.

In [ ]:
!hdfs dfs -mkdir -p /tp1/input
!hdfs dfs -rm -f /tp1/input/logs_300Mo.bin
!hdfs dfs -Ddfs.blocksize=134217728 -Ddfs.replication=3 -put /content/tp1/logs_300Mo.bin /tp1/input/
!hdfs dfs -ls -h /tp1/input | tee /content/preuves_tp/ls_tp1.txt

## Cellule 8 - Observer les blocs et la sous-réplication
**Attendu** : 3 blocs logiques ; `len=134217728`, `len=134217728`, `len=46137344` ; `Live_repl=1` sur chaque ligne ;
`Under-replicated blocks: 3 (100.0 %)`, `Missing replicas: 6`.

In [ ]:
!hdfs dfs -stat "Nom=%n Taille=%b Replication=%r" /tp1/input/logs_300Mo.bin | tee /content/preuves_tp/stat.txt
!hdfs fsck /tp1/input/logs_300Mo.bin -files -blocks -locations 2>/dev/null | tee /content/preuves_tp/fsck_300Mo.txt

## Cellule 9 - Vérifier la lecture et l'intégrité
**Attendu** : les deux empreintes sont identiques.

In [ ]:
!md5sum /content/tp1/logs_300Mo.bin | tee /content/preuves_tp/md5.txt
!hdfs dfs -cat /tp1/input/logs_300Mo.bin | md5sum | tee -a /content/preuves_tp/md5.txt

## Cellule 10 - Nettoyer
⚠️ La VM Colab et son stockage sont temporaires. Exécuter la cellule 11 **avant** si vous voulez garder les preuves.

In [ ]:
!hdfs dfs -rm -r -f /tp1
!rm -rf /content/tp1

## Cellule 11 - Copier les preuves dans l'espace partagé

In [ ]:
import shutil, os
SHARED_DIR = "/content/drive/MyDrive/TP_Hadoop_partage"
if os.path.isdir("/content/drive/MyDrive"):
    dest = os.path.join(SHARED_DIR, "preuves_tp")
    shutil.copytree("/content/preuves_tp", dest, dirs_exist_ok=True)
    print("Preuves copiées dans", dest)
    print(os.listdir(dest))
else:
    print("Drive non monté : exécuter la cellule 'Espace partagé' en haut du notebook.")

## Travail demandé dans Colab - Réponses

**1. Pourquoi le fichier de 300 Mo produit-il trois blocs ?**
HDFS découpe tout fichier en blocs de taille fixe `dfs.blocksize` = 128 Mo (134 217 728 octets).
300 Mo = 314 572 800 octets ; 314 572 800 ÷ 134 217 728 = 2,34375, donc ⌈2,34375⌉ = **3 blocs** : deux blocs pleins
et un troisième bloc partiel. Le dernier bloc n'occupe sur disque que sa taille réelle (44 Mo) et non 128 Mo.

**2. Longueurs exactes affichées par fsck**

| Bloc | len (octets) | Taille | Live_repl |
|---|---|---|---|
| 0 | 134 217 728 | 128 Mo | 1 |
| 1 | 134 217 728 | 128 Mo | 1 |
| 2 | 46 137 344 | 44 Mo | 1 |
| **Total** | **314 572 800** | **300 Mo** | |

**3. Réplication demandée et copies réellement disponibles**
La réplication **demandée** (3) est une métadonnée enregistrée par le NameNode pour le fichier, visible dans `hdfs dfs -ls`
(2ᵉ colonne) et dans `-stat %r`. Les copies **réelles** (`Live_repl=1`) dépendent du nombre de DataNodes, et HDFS ne place
jamais deux copies d'un même bloc sur le même DataNode. Avec un seul DataNode, il manque donc 2 copies par bloc :
`Under-replicated blocks: 3 (100 %)` et `Missing replicas: 6 (66,67 %)`. Le statut reste `HEALTHY`, car chaque bloc
a au moins une copie (réplication minimale = 1) et le fichier reste lisible (md5 identiques). Le volume physique réel est
de 300 Mo au lieu des 900 Mo attendus avec 3 copies.

**4. Pourquoi ne pas arrêter le seul DataNode de Colab ?**
Ce DataNode détient l'**unique** copie de chaque bloc. Si on l'arrête, les 3 blocs deviennent *missing*, le fichier devient
illisible (`fsck` → `CORRUPT`) et le NameNode ne peut rien re-répliquer, faute d'autre DataNode cible. On ne démontrerait pas
la tolérance aux pannes, seulement une perte de disponibilité. On risquerait aussi un safe mode qui bloquerait la suite du TP.
La vraie simulation de panne nécessite plusieurs DataNodes (Partie B).

**5. Comparaison avec le calcul théorique du fichier de 1 200 Mo**

| | Colab (300 Mo) | Théorie (1 200 Mo, 6 DataNodes) |
|---|---|---|
| Taille | 314 572 800 o | 1 258 291 200 o |
| Calcul | 300 / 128 = 2,34 | 1 200 / 128 = 9,375 |
| Blocs logiques | 3 | 10 |
| Dernier bloc | 44 Mo (46 137 344 o) | 48 Mo (50 331 648 o) |
| Réplication demandée | 3 | 3 |
| Copies physiques | 3 (1 par bloc) | 30 (3 par bloc) |
| Volume physique | 300 Mo | 3 600 Mo |
| Tolérance à la perte d'un nœud | Aucune | Oui (2 copies restantes) |

La logique de découpage est identique et le calcul se vérifie en pratique. Seule la réplication effective diffère :
elle est limitée par le nombre de DataNodes (min(3, 1) = 1 dans Colab, 3 avec 6 DataNodes).